# Практика 10. Неперервні розподіли в scipy.stats

**Варіант 1: Київ**

Мета заняття: підібрати параметри нормального розподілу (`μ` і `σ`) під власну вибірку даних через середнє й стандартне відхилення, а потім обчислити `.pdf()`, `.cdf()` і `.ppf()` для `scipy.stats.norm`.

У цій роботі я використовую липневу середню температуру Києва: `21 °C`.

In [1]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

np.random.seed(1)
daily_temps = np.random.normal(loc=21, scale=2.5, size=30)
print("daily_temps =", daily_temps)
print("mean =", daily_temps.mean())
print("std =", daily_temps.std())

daily_temps = [25.06086341 19.47060897 19.67957062 18.31757844 23.16351907 15.24615326
 25.36202941 19.09698275 21.79759774 20.37657406 24.65526984 15.84964823
 20.19395699 20.03986411 23.83442361 18.25027183 20.56892948 18.80535396
 21.10553437 22.45703803 18.24845206 23.86180927 23.2539768  22.25623585
 23.25213987 19.29068035 20.69277444 18.66057641 20.3302798  22.32588867]
mean = 20.850152723360946
std = 2.5227852196402956


## Завдання 1. Побудова вибірки денних температур

Згідно з правилом практики, для варіанта 1 беремо липень Києва: `21 °C`. Генеруємо 30 денних температур навколо цього центру з `scale=2.5`:

```python
np.random.seed(1)
daily_temps = np.random.normal(loc=21, scale=2.5, size=30)
```

У масиві `daily_temps` середнє близьке до 21 °C, але не рівне йому точно, бо це випадкова вибірка. Середнє та стандартне відхилення наведені у попередній клітинці.

In [2]:
# Підготовлені дані для подальших обчислень
mean = daily_temps.mean()
std = daily_temps.std()
print(f"Середнє вибірки: {mean:.4f} °C")
print(f"Стандартне відхилення: {std:.4f} °C")
print(f"Відхилення від липня: {mean - 21:.4f} °C")

Середнє вибірки: 20.8502 °C
Стандартне відхилення: 2.5228 °C
Відхилення від липня: -0.1498 °C


## Завдання 2. Параметри нормального розподілу

1. Обчислюємо `mean` і `std` вибірки `daily_temps`.
2. Створюємо теоретичний нормальний розподіл `stats.norm(loc=mean, scale=std)`.

Це і є модель, підібрана саме під цю вибірку.

In [ ]:
mean = daily_temps.mean()
std = daily_temps.std()
dist = stats.norm(loc=mean, scale=std)

print("Параметри нормального розподілу:")
print("mu =", dist.mean())
print("sigma =", dist.std())
print("Формально: stats.norm(loc=mean, scale=std)")

## Завдання 3. Гістограма і теоретична pdf-крива

Побудуємо гістограму реальної вибірки з `density=True` і накладемо на неї криву щільності `.pdf(x)` для підібраного нормального розподілу.

In [ ]:
# Діапазон для побудови pdf
x = np.linspace(daily_temps.min() - 5, daily_temps.max() + 5, 200)

plt.figure(figsize=(10, 6))
plt.hist(daily_temps, bins=8, density=True, edgecolor='white', alpha=0.75, label='Вибірка денних температур')
plt.plot(x, dist.pdf(x), color='crimson', linewidth=2, label='Нормальна pdf, fitted')
plt.title('Гістограма температур та теоретична pdf-крива')
plt.xlabel('Температура, °C')
plt.ylabel('Густина')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Оцінка на око

Крива `pdf(x)` проходить близько до вершин стовпців гістограми: форма дзвоноподібна, а центр знаходиться поруч із середнім значенням вибірки. Помітних відхилень форми мало, що є нормальним для 30 випадкових спостережень. Невелика різниця пояснюється тим, що вибірка є конкретною реалізацією випадкового процесу, а не ідеальною теоретичною нормальною популяцією.

## Завдання 4. Ймовірність потрапляння в діапазон

Застосуємо `.cdf()` для діапазону `[mean - std, mean + std]`:

`P(mean - std <= X <= mean + std) = CDF(mean + std) - CDF(mean - std)`

Це приблизно 68%, бо для нормального розподілу в межах одного стандартного відхилення від середнього лежить близько 68.27% всіх значень.

In [ ]:
prob_within_1sigma = dist.cdf(mean + std) - dist.cdf(mean - std)
print(f"Ймовірність потрапляння в [mean - std, mean + std] = {prob_within_1sigma:.4f}")
print(f"У відсотках: {prob_within_1sigma * 100:.2f}%")
print("Правило 68-95-99.7 для нормального розподілу: близько 68.27% в межах 1σ.")

### Пояснення

Отримане число буде близьке до `0.6827`, тобто ~68%, але не завжди буде точним саме 68% через випадкову вибірку з 30 значень. Теоретична норма дає 68.27%, а конкретна вибірка може відхилитися в той чи інший бік. Саме тому невелика різниця є цілком природною.

## Завдання 5. Квантиль через `ppf`

Знайдемо температуру, що має 95-й процентиль для підібраного нормального розподілу:

In [ ]:
q95 = dist.ppf(0.95)
print(f"95-й процентиль: {q95:.4f} °C")

### Пояснення

`dist.ppf(0.95)` повертає значення температури, нижче якої лежить 95% значень розподілу. Для нашої моделі це приблизно `q95` °C: 95% денних температур у цій нормальній моделі не перевищують цю температуру, а 5% залишаються вище. Тобто `.ppf()` є оберненою операцією до `.cdf()`: `cdf(x)` відповідає на питання “яка ймовірність того, що значення не більше x?”, а `ppf(p)` — “яке x відповідає накопиченій ймовірності p?”

## Короткі письмові відповіді

1. `pdf(x)` повертає щільність ймовірності в точці `x`, тобто “як густо розподіл розташований навколо цього значення”. `cdf(x)` повертає накопичену ймовірність: “яка ймовірність, що випадкова величина не перевищить `x`?”

2. Значення `.pdf(x)` може бути більшим за 1, бо це не ймовірність, а густина ймовірності. Для неперервних розподілів ймовірність на окремій точці дорівнює нулю, а щільність може бути будь-якою додатною величиною. Вона не суперечить тому, що ймовірність не перевищує 1, бо ймовірність обчислюється як площа під кривою на інтервалі, а не як значення самої функції в точці.

3. `.ppf()` називають оберненою до `.cdf()`, бо `.cdf(p)` приймає значення `x` і повертає накопичену ймовірність, а `.ppf(q)` приймає ймовірність `q` і повертає відповідне значення `x`. Іншими словами, `cdf(ppf(q)) = q` для відповідних значень, а `ppf(cdf(x)) = x`.

## Підсумки

- Для Києва взято липень `21 °C` як орієнтовний центр.
- Вибірка з 30 денних температур згенерована через `np.random.normal(loc=21, scale=2.5, size=30)`.
- Для вибірки підібрано нормальний розподіл `stats.norm(loc=mean, scale=std)`.
- Гістограма й `pdf(x)` накладені на одному графіку з `density=True`.
- Через `.cdf()` обчислено ймовірність потрапляння в інтервал `[mean - std, mean + std]`.
- Через `.ppf()` знайдено 95-й процентиль.